# 📘 Notebook 2 — I dati: caricamento, esplorazione e preparazione

Nel notebook precedente abbiamo visto la matematica di base. Ora ci occupiamo del **carburante del Machine Learning**: i **dati**.

## 🎯 Cosa imparerai qui
1. Cos'è un **dataset** e come è strutturato
2. Come usare **pandas** per caricare ed esplorare i dati
3. Come **visualizzare** i dati per capirli
4. Come **pulire** i dati (valori mancanti, valori strani)
5. Come **trasformare** i dati: scaling, encoding, train/test split
6. Perché la preparazione dei dati è l'80% del lavoro!

## 🤔 Perché è così importante?
> *"Garbage in, garbage out"* — se dai dati spazzatura al modello, otterrai previsioni spazzatura.

Anche il miglior modello del mondo non può fare miracoli con dati mal preparati.

## 1️⃣ Cos'è un dataset?

Un **dataset** è una tabella, dove:
- ogni **riga** è un **esempio** (es: una persona, una casa, un'email)
- ogni **colonna** è una **caratteristica** (in inglese *feature*) di quell'esempio
- una colonna speciale è il **target** (o *label*): quello che vogliamo prevedere

Esempio: un dataset di case con `superficie`, `numero_stanze`, `quartiere`, `prezzo` (target).

In [ ]:
# Importiamo pandas: la libreria standard per maneggiare tabelle in Python
import pandas as pd
import numpy as np

# Creiamo un piccolo dataset "a mano" per capire la struttura
dati = {
    'superficie_mq': [50, 80, 120, 65, 95, 200, 45, 110],
    'stanze':        [2,  3,  4,   2,  3,  5,   1,  4  ],
    'quartiere':     ['centro', 'periferia', 'centro', 'periferia', 'semicentro', 'centro', 'periferia', 'semicentro'],
    'prezzo_euro':   [200000, 180000, 450000, 150000, 280000, 800000, 120000, 380000]
}

# Convertiamo il dizionario in un DataFrame (la "tabella" di pandas)
df = pd.DataFrame(dati)
df  # mostriamo il dataframe (in Jupyter compare con bella formattazione)

## 2️⃣ Esplorazione del dataset

![schema dati](Media/02/dataset.png)
Prima di costruire qualunque modello, dobbiamo **conoscere** i nostri dati. Le prime cose da guardare sempre:

In [1]:
# Forma del dataset (righe, colonne)
print("Forma:", df.shape)

# Tipi di dato per ogni colonna
print("\nTipi di dato:")
print(df.dtypes)
# - int64 / float64 = numeri
# - object = testo (stringhe)

# Le prime 5 righe
print("\nPrime righe:")
print(df.head())

# Statistiche descrittive (solo per colonne numeriche)
print("\nStatistiche:")
print(df.describe())
# count = quanti valori non vuoti
# mean = media,  std = deviazione standard
# min/25%/50%/75%/max = i quartili (utili per capire la distribuzione)

NameError: name 'df' is not defined

### Selezionare colonne e righe

In [ ]:
# Selezionare una colonna (restituisce una Series)
print("Colonna prezzo:")
print(df['prezzo_euro'])

# Selezionare più colonne (restituisce un DataFrame)
print("\nDue colonne:")
print(df[['superficie_mq', 'prezzo_euro']])

# Filtrare le righe in base a una condizione
case_grandi = df[df['superficie_mq'] > 100]
print("\nSolo le case sopra i 100 mq:")
print(case_grandi)

# 🔧 PROVA TU: trova solo le case in centro

## 3️⃣ Visualizzare i dati

Un grafico vale più di mille numeri! Visualizzare permette di **scoprire pattern** invisibili nelle tabelle.

In [ ]:
import matplotlib.pyplot as plt

# Scatter plot: punti su un piano. Asse X = superficie, asse Y = prezzo
plt.figure(figsize=(8, 5))
plt.scatter(df['superficie_mq'], df['prezzo_euro'], color='blue', s=80)
plt.xlabel('Superficie (mq)')
plt.ylabel('Prezzo (€)')
plt.title('Relazione tra superficie e prezzo')
plt.grid(True, alpha=0.3)
plt.show()

# 💡 Vedi che il prezzo cresce all'aumentare della superficie?
# Si dice che le due variabili sono "correlate positivamente".

In [ ]:
# Istogramma: come si distribuiscono i prezzi?
plt.figure(figsize=(8, 4))
plt.hist(df['prezzo_euro'], bins=8, color='salmon', edgecolor='black')
plt.xlabel('Prezzo (€)')
plt.ylabel('Frequenza')
plt.title('Distribuzione dei prezzi')
plt.show()

In [ ]:
# Boxplot: prezzi per quartiere (utile per confrontare gruppi)
df.boxplot(column='prezzo_euro', by='quartiere', figsize=(8, 5))
plt.title('Prezzi per quartiere')
plt.suptitle('')  # rimuove il titolo automatico di matplotlib
plt.ylabel('Prezzo (€)')
plt.show()

# 💡 Il boxplot mostra:
# - la linea centrale = mediana
# - la "scatola" = dove sta il 50% dei dati
# - i "baffi" = il range più ampio
# - i pallini = valori anomali (outlier)

## 4️⃣ Pulizia dei dati

I dati **reali** sono sempre sporchi: valori mancanti, errori di battitura, valori impossibili...

Aggiungiamo un po' di "sporcizia" per imparare a gestirla:

In [ ]:
# Creiamo un dataset "sporco"
dati_sporchi = {
    'superficie_mq': [50, 80, np.nan, 65, 95, 200, 45, 9999],   # un NaN (valore mancante) e un valore assurdo
    'stanze':        [2,  3,  4,   2,  3,  5,   1,  4  ],
    'quartiere':     ['centro', 'Periferia', 'CENTRO', 'periferia', 'semicentro', None, 'periferia', 'semicentro'],
    'prezzo_euro':   [200000, 180000, 450000, 150000, np.nan, 800000, 120000, 380000]
}
df_sporco = pd.DataFrame(dati_sporchi)
df_sporco

In [ ]:
# Quanti valori mancanti per colonna?
print("Valori mancanti per colonna:")
print(df_sporco.isnull().sum())

# Strategia 1: ELIMINARE le righe con valori mancanti (drop)
# Veloce ma perdiamo dati
df_drop = df_sporco.dropna()
print(f"\nDopo drop: {len(df_drop)} righe (su {len(df_sporco)})")

# Strategia 2: RIEMPIRE i valori mancanti (imputation)
# Più intelligente: usiamo media o moda
df_pulito = df_sporco.copy()

# Per le colonne numeriche → riempiamo con la media
df_pulito['superficie_mq'] = df_pulito['superficie_mq'].fillna(df_pulito['superficie_mq'].median())
df_pulito['prezzo_euro']   = df_pulito['prezzo_euro'].fillna(df_pulito['prezzo_euro'].median())

# Per le colonne testuali → riempiamo con la moda (valore più frequente)
df_pulito['quartiere'] = df_pulito['quartiere'].fillna(df_pulito['quartiere'].mode()[0])

print("\nDopo imputazione:")
print(df_pulito.isnull().sum())

In [ ]:
# Sistemiamo le maiuscole/minuscole (CENTRO vs centro vs Centro sono la stessa cosa!)
df_pulito['quartiere'] = df_pulito['quartiere'].str.lower().str.strip()
print("Quartieri unici dopo pulizia:", df_pulito['quartiere'].unique())

# Gestione degli OUTLIER (valori anomali)
# Una casa di 9999 mq è chiaramente un errore!
# Strategia: rimuovere i valori troppo lontani dalla media

# Calcoliamo i limiti accettabili
media = df_pulito['superficie_mq'].mean()
std = df_pulito['superficie_mq'].std()
soglia_alta = media + 3 * std   # regola empirica: oltre 3 deviazioni standard è sospetto

print(f"\nSoglia outlier: {soglia_alta:.0f} mq")
df_pulito = df_pulito[df_pulito['superficie_mq'] < soglia_alta]
print("\nDataset pulito:")
print(df_pulito)

## 5️⃣ Trasformazione dei dati per il modello

I modelli di ML lavorano **solo con numeri**. Quindi dobbiamo:
1. **Encoding**: convertire le categorie testuali in numeri
2. **Scaling**: portare tutte le colonne sulla stessa scala (importantissimo!)
3. **Train/Test split**: dividere i dati in "per allenarsi" e "per testare"

### 5.1 Encoding delle variabili categoriche

La colonna `quartiere` ha testo. Dobbiamo trasformarla in numeri.

![schema encoding](Media/02/categorie_numeri.png)


In [2]:
# Riprendiamo il dataset originale pulito
df = pd.DataFrame({
    'superficie_mq': [50, 80, 120, 65, 95, 200, 45, 110],
    'stanze':        [2,  3,  4,   2,  3,  5,   1,  4  ],
    'quartiere':     ['centro', 'periferia', 'centro', 'periferia', 'semicentro', 'centro', 'periferia', 'semicentro'],
    'prezzo_euro':   [200000, 180000, 450000, 150000, 280000, 800000, 120000, 380000]
})

# Tecnica: ONE-HOT ENCODING
# Crea una colonna 0/1 per ogni categoria
# (è più sicuro del semplice "centro=1, periferia=2, semicentro=3"
# perché questi numeri non implicano un ordinamento)
df_encoded = pd.get_dummies(df, columns=['quartiere'], dtype=int)
df_encoded

NameError: name 'pd' is not defined

### 5.2 Scaling (normalizzazione)

**Problema**: la superficie va da 45 a 200, il prezzo da 120000 a 800000. Le scale sono molto diverse!

Molti algoritmi di ML "si confondono" se le caratteristiche hanno scale diverse e finiscono per dare più peso alle caratteristiche con numeri più grandi.

**Soluzione**: portare tutto sulla stessa scala. Due tecniche comuni:
- **MinMaxScaler**: porta tutti i valori tra 0 e 1
- **StandardScaler**: rende media=0 e deviazione standard=1 ("standardizzazione")

![schema scaling](Media/02/scaling.png)


In [3]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
# scikit-learn (sklearn) è LA libreria di ML in Python

# Selezioniamo solo le colonne numeriche da scalare
colonne_numeriche = ['superficie_mq', 'stanze']
X = df_encoded[colonne_numeriche].copy()

print("Prima dello scaling:")
print(X.describe().round(2))

# MinMax: porta tra 0 e 1
scaler_minmax = MinMaxScaler()
X_minmax = scaler_minmax.fit_transform(X)
# fit_transform = "impara" min e max e poi trasforma
X_minmax_df = pd.DataFrame(X_minmax, columns=colonne_numeriche)
print("\nDopo MinMaxScaler (tutto tra 0 e 1):")
print(X_minmax_df.round(3))

# Standard: media=0, std=1
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X)
X_std_df = pd.DataFrame(X_std, columns=colonne_numeriche)
print("\nDopo StandardScaler (media=0, std=1):")
print(X_std_df.round(3))

# 💡 Quale usare?
# - MinMax: quando vuoi un range fisso (es: reti neurali con sigmoid)
# - Standard: il default per la maggior parte dei casi
# - Nessuno: per alcuni modelli (es: alberi decisionali) non serve!

ModuleNotFoundError: No module named 'sklearn'

### 5.3 Train/Test split — il concetto più importante!

🎓 **Analogia**: se uno studente si esercita sempre sulle stesse 10 domande e poi all'esame gliene fanno una nuova, magari va male. Per valutare se ha **veramente** imparato, dobbiamo testarlo su domande **mai viste prima**.

Stesso principio per i modelli di ML:
- **Training set** (es: 80% dei dati): per **allenare** il modello
- **Test set** (es: 20% dei dati): per **valutare** il modello su dati nuovi

![schema trainig](Media/02/traing_test.png)


Se il modello va bene sul training ma male sul test → si dice che fa **overfitting** (ha "memorizzato" invece di imparare).

![schema trainig](Media/02/overfitting.png)


Lo vedremo nel prossimo notebook!

In [5]:
from sklearn.model_selection import train_test_split

# X = caratteristiche (input), y = target (output da prevedere)
X = df_encoded.drop('prezzo_euro', axis=1)   # tutte le colonne TRANNE il prezzo
y = df_encoded['prezzo_euro']                # solo la colonna prezzo

# Dividiamo in train e test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,    # 25% al test, 75% al train
    random_state=42    # per la riproducibilità (la "divisione" sarà sempre uguale)
)

print(f"Dati totali:    {len(X)}")
print(f"Training set:   {len(X_train)} esempi")
print(f"Test set:       {len(X_test)} esempi")

print("\nX_train:")
print(X_train)
print("\ny_train:")
print(y_train.values)

ModuleNotFoundError: No module named 'sklearn'

## 🎓 Riepilogo del Notebook 2

| Step | Cosa abbiamo fatto |
|---|---|
| **Esplorazione** | `.head()`, `.describe()`, `.dtypes` |
| **Visualizzazione** | scatter, istogrammi, boxplot |
| **Pulizia** | `dropna`, `fillna`, gestione outlier |
| **Encoding** | `pd.get_dummies` per categorie testuali |
| **Scaling** | `StandardScaler` o `MinMaxScaler` |
| **Split** | `train_test_split` per separare allenamento e test |

### ⚠️ Regola d'oro
Nei progetti reali **almeno il 70-80% del tempo** si passa qui (sui dati), non sui modelli.

👉 **Prossimo notebook (03)**: il primo vero modello di ML — la **Regressione Lineare** — con cui prevediamo il prezzo delle case!